# R3_07 — Analisis router (tanpa panggilan API baru)

Dengan **anggaran yang sama** dengan router utama (jumlah pasangan yang ditandai), bandingkan empat cara memilih pasangan yang dikirim
ke LLM:

* (a) router utama — vote terpecah dulu, lalu margin soft vote terkecil (pada anggaran = jumlah ditandai, ini persis aturan NB06);
* (b) confidence IndoBERT terendah (probabilitas maks rata-rata 3 seed);
* (c) confidence weighted soft voting terendah;
* (d) entropi weighted soft voting tertinggi.

Pasangan yang dirouting memakai label **LLM-always** (R3_05, prompt baseline per pasangan, sudah ada untuk semua pasangan → tidak ada
panggilan baru); sisanya memakai label weighted soft voting. Dilaporkan: kesalahan soft voting yang tertangkap, yang diperbaiki/dirusak
LLM, akurasi, ASC F1, Overall F1 (CI 95% cluster bootstrap). Anggaran tambahan: 140 dan 200.

`DRY_RUN = True` (default) memakai keluaran dry (dev). Mode nyata memakai label uji dan hanya dijalankan setelah R3_06.

In [ ]:
# ── PARAMETER ──
DRY_RUN = True
BUDGETS_EXTRA = [140, 200]

import os
DRY_RUN = os.environ.get('R3_DRY_RUN', str(DRY_RUN)) == 'True'

In [ ]:
import sys, json, warnings
sys.path.insert(0, os.environ.get('R3_CODE_DIR', os.getcwd()))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import config as C
import r3_lib as L
import r3_eval as E
ru = E.ru
warnings.filterwarnings('ignore')
D = L.mode_dirs(DRY_RUN)
if DRY_RUN:
    L.forbid_test_access(allow_dirs=[C.DRY_DIR])
frozen = L.verify_frozen(strict=not DRY_RUN, check_models=False)
for need in [D.test_pred / 'DONE.json', D.llm / 'DONE.json']:
    assert need.exists(), f'Belum ada: {need}'
if not DRY_RUN:
    assert (D.eval / 'DONE.json').exists(), 'Jalankan R3_06 (evaluasi utama) lebih dulu'
    assert not json.loads((D.llm / 'DONE.json').read_text())['mock_used'], 'label LLM mock tidak boleh dipakai di mode nyata'
OUT = C.assert_writable(D.router); OUT.mkdir(parents=True, exist_ok=True)
SAVE = lambda df, name, title='', index=False: ru.save_table(df, OUT, name, title, index=index)
print(f"Mode: {'DRY_RUN (label dev)' if DRY_RUN else 'NYATA (label uji)'} | output: {OUT}")

In [ ]:
df, probs = E.build_eval_frame(D, frozen)
n_flag = int(df.flagged.sum())
budgets = [n_flag] + [b for b in BUDGETS_EXTRA if b != n_flag]
over = [b for b in budgets if b > len(df)]
assert not over, f'Anggaran melebihi jumlah pasangan ({len(df)}): {over}'
rank = E.router_rankings(df, probs['ensemble'], probs['indobert'])
key_a = next(k for k in rank if k.startswith('(a)'))
assert set(rank[key_a][:n_flag]) == set(np.flatnonzero(df.flagged.values)), 'router (a) pada anggaran utama ≠ himpunan ditandai'
print(f'{len(df)} pasangan | anggaran utama = {n_flag} (= jumlah ditandai) | anggaran: {budgets}')
print('✓ Router (a) pada anggaran utama mereproduksi himpunan pasangan yang ditandai')

In [ ]:
tab = E.router_eval(df, rank, budgets)
ref, _ = E.metrics_ci(df, ['Weighted Soft Voting', 'Soft Voting + LLM', 'LLM-always'])
ref.insert(0, 'catatan', ['tanpa routing', 'hybrid utama (arbitrase retain/revise)', 'LLM untuk semua pasangan'])
SAVE(tab, 'T_router_perbandingan', 'Router alternatif: pasangan dirouting memakai label LLM-always; CI 95% cluster bootstrap')
SAVE(ref, 'T_router_referensi', 'Referensi: tanpa routing, hybrid utama, LLM-always')
show = tab.assign(**{k: [E.fmt_ci(e, lo, hi) for e, lo, hi in zip(tab[k], tab[f'{k}_lo'], tab[f'{k}_hi'])]
                     for k in ['accuracy', 'asc_f1', 'overall_f1']})
show[['anggaran', 'router', 'n_dirouting', 'kesalahan_soft_tertangkap', 'diperbaiki_LLM', 'dirusak_LLM', 'accuracy', 'asc_f1', 'overall_f1']]

In [ ]:
# Overlap himpunan yang dirouting dengan router utama (anggaran utama)
sets = {k: set(v[:n_flag]) for k, v in rank.items()}
ov = pd.DataFrame([{'router': k, f'irisan dengan (a) pada anggaran {n_flag}': len(s & sets[key_a]),
                    'Jaccard': len(s & sets[key_a]) / len(s | sets[key_a])} for k, s in sets.items()])
SAVE(ov, 'T_router_overlap', 'Irisan pasangan yang dirouting dengan router utama')

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, k in zip(axes, ['accuracy', 'asc_f1', 'overall_f1']):
    for r in rank:
        t = tab[tab.router == r].sort_values('anggaran')
        ax.errorbar(t.anggaran, t[k], yerr=[t[k] - t[f'{k}_lo'], t[f'{k}_hi'] - t[k]], marker='o', capsize=3, label=r.split(')')[0] + ')')
    ax.axhline(float(ref.loc[ref.skema == 'Weighted Soft Voting', k].iloc[0]), color='#999', ls='--', lw=1)
    ax.set_title(E.METRIC_LABEL[k]); ax.set_xlabel('anggaran (pasangan ke LLM)'); ax.set_xticks(budgets)
axes[0].legend(fontsize=8, frameon=False)
plt.tight_layout(); fig.savefig(OUT / 'router_perbandingan.png', dpi=200, bbox_inches='tight'); plt.close(fig)
L.write_json(OUT / 'DONE.json', {'finished_at': L.now_utc(), 'dry_run': DRY_RUN, 'budgets': budgets, 'n_pairs': len(df),
                                 'new_api_calls': 0})
print('✓ Selesai:', OUT)